<a href="https://colab.research.google.com/github/areslou/cpecog2-coral-bleaching/blob/main/notebooks/Group2_Coral_Run7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group 2 CPECOG2: Coral bleaching, Run 7 (automatic label cleaning, 3-seed averaging, full evaluation)

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all (about 45 to 60 min).

Run 6 showed that larger backbones and inputs do not help: every candidate plateaued at a cross-validated ROC AUC of about 0.89, and in every fold all three models confidently misclassified the same 77 photos. Run 7 keeps the proposal's model (YOLOv8s-cls, Run 5 R14 settings, 224 px) and tests two changes that need no manual labeling, on the same 5 folds as Run 6:

| Config | What changes | Why |
| :-- | :-- | :-- |
| A | nothing (Run 6 C1, re-run here) | the reference |
| B | **automatic label cleaning of the training photos only** (confident learning) | some folder labels are likely wrong; a model trained on fewer wrong labels can rank better |
| +3 seeds | the winner of A and B, averaged over seeds 42, 43 and 44 | averaging Run 6's three models already raised ROC AUC from 0.894 to 0.904 |

**How the cleaning works, and why it is fair:** inside each fold, a quick proxy model (logistic regression on frozen ImageNet YOLOv8s-cls features) scores only that fold's training photos with its own inner 5-fold cross-validation. A training photo is dropped when the proxy confidently assigns it to the other class (the confident-learning rule: its other-class probability is above that class's average self-confidence), with at most 10% of the training photos dropped. The held-out fold and the test set are never cleaned or looked at, so their scores stay honest.

Everything in the proposal is kept: the scratch CNN baseline (Objective 1), the held-out test set evaluated once, mean and standard deviation over 3 seeds, the external **BU + BHD** test set (BHD files are named `hbd` in the zip, the reason Run 2 missed them), and the size and CPU-latency limits. The operating cutoff is chosen from out-of-fold predictions with the Run 3 rule (highest macro F1 with bleached recall of at least 0.90); 0.5 is also reported.

Outputs go to `results/run7/` and are zipped as `run7_results.zip`.

In [ ]:
!pip -q install ultralytics==8.4.211 imagehash kagglehub huggingface_hub

In [ ]:
import os, glob, shutil, time, json, random, math, copy, re, zipfile
import numpy as np, pandas as pd
from PIL import Image
import imagehash, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, roc_auc_score, roc_curve, confusion_matrix

SEED = 42
FOLDS = 5
EPOCHS = 20                  # fixed epochs for every YOLO fit; no early stopping, the last epoch is used
CNN_EPOCHS = 40
BATCH = 32
IMGSZ = 224
SEEDS = [42, 43, 44]
RECALL_MIN = 0.90            # Run 3 cutoff rule
MAX_PRUNE = 0.10             # at most 10% of a training set is dropped by the label cleaning
MAX_MB, MAX_MS = 50, 200
NEAR_DUP_T = 5
DATA_SRC = None              # set a folder path here to skip the Kaggle download
EXT_SRC = None               # set a .zip path here to skip the Hugging Face download
SPLITS_URL = "https://raw.githubusercontent.com/areslou/cpecog2-coral-bleaching/main/data/splits.csv"
YOLO_AUG = dict(dropout=0.3, auto_augment=None, erasing=0.0, scale=0.25, fliplr=0.5, flipud=0.5, hsv_h=0.0, hsv_s=0.2, hsv_v=0.1)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
TDEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU:", GPU_NAME if torch.cuda.is_available() else "none. Runtime > Change runtime type > T4 GPU, or this will take hours.")
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
OUT = os.path.join(BASE, "results", "run7"); os.makedirs(OUT, exist_ok=True)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
T_START = time.time()

## 1. Data: the cleaned split and the Run 6 folds

Same code as Run 6: the Run 1 split minus the conflicting-label photos and the cross-split near-duplicate; train + validation pooled (761 images) into the same 5 stratified folds (seed 42); the 131 test images stay aside.

In [ ]:
if DATA_SRC is None:
    import kagglehub
    DATA_SRC = kagglehub.dataset_download("vencerlanz09/healthy-and-bleached-corals-image-classification")
exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def label_of(p):
    parent = os.path.basename(os.path.dirname(p)).lower()
    if "bleach" in parent: return "bleached"
    if "health" in parent: return "healthy"
    return None

def phash_of(p):
    try:
        with Image.open(p) as im:
            return str(imagehash.phash(im.convert("RGB")))
    except Exception:
        return None

POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def hamming(a, b):
    x = np.array([int(h, 16) for h in a], dtype=np.uint64)[:, None] ^ np.array([int(h, 16) for h in b], dtype=np.uint64)[None, :]
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x).astype(np.int16)
    return POP[np.ascontiguousarray(x).view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(-1).astype(np.int16)

paths = sorted(p for p in glob.glob(os.path.join(DATA_SRC, "**", "*"), recursive=True) if p.lower().endswith(exts))
df = pd.DataFrame({"path": paths})
df["label"] = df["path"].map(label_of)
df["file"] = df["path"].map(os.path.basename)
df["phash"] = df["path"].map(phash_of)
df = df.dropna(subset=["label", "phash"]).reset_index(drop=True)

splits1 = pd.read_csv(SPLITS_URL).rename(columns={"path": "file"})
df = df.merge(splits1[["file", "label", "split"]], on=["file", "label"], how="left")
assert int(df["split"].notna().sum()) == len(splits1), "splits.csv did not match the download"

n_lab = df.groupby("phash")["label"].nunique()
conflict_hashes = set(n_lab[n_lab > 1].index)
clean = df[df["split"].notna() & ~df["phash"].isin(conflict_hashes)].reset_index(drop=True)
D = hamming(clean["phash"], clean["phash"])
sp, lb = clean["split"].to_numpy(), clean["label"].to_numpy()
ii, jj = np.where(np.triu(D <= NEAR_DUP_T, k=1))
rank = {"train": 0, "val": 1, "test": 2}
drop = {max((i, j), key=lambda k: rank[sp[k]]) for i, j in zip(ii, jj) if sp[i] != sp[j] and lb[i] == lb[j]}
drop |= {k for i, j in zip(ii, jj) if lb[i] != lb[j] for k in (i, j)}
clean = clean.drop(index=sorted(drop)).reset_index(drop=True)
print(f"Removed {len(conflict_hashes)} conflicting-label groups and {len(drop)} near-duplicates")
print(pd.crosstab(clean["split"], clean["label"], margins=True))

In [ ]:
def pad_square(p):
    im = Image.open(p).convert("RGB"); s = max(im.size)
    sq = Image.new("RGB", (s, s), (0, 0, 0)); sq.paste(im, ((s - im.size[0]) // 2, (s - im.size[1]) // 2))
    return sq

POOL = os.path.join(BASE, "coral_pool"); shutil.rmtree(POOL, ignore_errors=True)
imgs = []
for _, r in clean.iterrows():
    d = os.path.join(POOL, r["label"]); os.makedirs(d, exist_ok=True)
    dst = os.path.join(d, os.path.splitext(r["file"])[0] + ".jpg")
    pad_square(r["path"]).save(dst, quality=95); imgs.append(dst)
clean["img"] = imgs
assert clean["img"].nunique() == len(clean), "file name clash in the image pool"

trval = clean[clean["split"] != "test"].reset_index(drop=True)
test = clean[clean["split"] == "test"].reset_index(drop=True)
y_trval = (trval["label"] == "bleached").astype(int).to_numpy()
y_test = (test["label"] == "bleached").astype(int).to_numpy()
test_files = test["img"].tolist()
FOLD_IDX = list(StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(trval)), y_trval))
fold_of = np.zeros(len(trval), dtype=int)
for k, (_, va) in enumerate(FOLD_IDX): fold_of[va] = k
trval.assign(fold=fold_of)[["file", "label", "split", "fold"]].to_csv(os.path.join(OUT, "folds.csv"), index=False)
test[["file", "label"]].to_csv(os.path.join(OUT, "test_images.csv"), index=False)
print(f"Train + validation: {len(trval)} ({y_trval.sum()} bleached) in {FOLDS} folds | test: {len(test)} ({y_test.sum()} bleached)")

## 2. Helpers: metrics, the cutoff rule, YOLO training, the scratch CNN, and the label cleaning

In [ ]:
def metrics_at(y, p, thr):
    yp = (np.asarray(p) >= thr).astype(int)
    return dict(macro_f1=f1_score(y, yp, average="macro"), bleached_recall=recall_score(y, yp, zero_division=0),
                bleached_precision=precision_score(y, yp, zero_division=0), healthy_recall=recall_score(y, yp, pos_label=0, zero_division=0),
                accuracy=accuracy_score(y, yp), roc_auc=roc_auc_score(y, p), cm=confusion_matrix(y, yp, labels=[0, 1]).tolist())

def pick_threshold(y, p):
    rows = [(t, f1_score(y, p >= t, average="macro"), recall_score(y, p >= t, zero_division=0)) for t in np.round(np.linspace(0.01, 0.99, 981), 3)]
    ok = [r for r in rows if r[2] >= RECALL_MIN]
    return max(ok or rows, key=lambda r: (r[1], r[0]))[0], bool(ok)

def make_folder(name, train_idx, val_idx):
    root = os.path.join(BASE, "cv", name); shutil.rmtree(root, ignore_errors=True)
    for part, idx in (("train", train_idx), ("val", val_idx)):
        for i in idx:
            r = trval.iloc[i]; d = os.path.join(root, part, r["label"]); os.makedirs(d, exist_ok=True)
            os.symlink(r["img"], os.path.join(d, os.path.basename(r["img"])))
    return root

from ultralytics import YOLO
def yolo_fit(data_root, name, seed):
    m = YOLO("yolov8s-cls.pt")
    m.train(data=data_root, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=seed, deterministic=True, patience=EPOCHS + 1,
            val=False, plots=False, project=os.path.join(BASE, "runs7"), name=name, exist_ok=True, device=DEVICE, verbose=False, **YOLO_AUG)
    return str(m.trainer.last)

def yolo_predict(weights, files, bs=64):
    m = YOLO(weights); b = next(k for k, n in m.names.items() if n == "bleached"); p = []
    for k in range(0, len(files), bs):
        p += [float(r.probs.data[b]) for r in m.predict(files[k:k + bs], imgsz=IMGSZ, device=DEVICE, verbose=False)]
    return np.array(p)

# label cleaning: proxy scores from frozen ImageNet features, computed only within the training photos it cleans
_emb_model = YOLO("yolov8s-cls.pt")
EMB = np.stack([e.cpu().numpy() for e in _emb_model.embed(trval["img"].tolist(), imgsz=IMGSZ, verbose=False)])
del _emb_model

def clean_training(idx):
    """Confident learning on the training indices idx; returns (kept indices, dropped indices, proxy ROC AUC)."""
    idx = np.asarray(idx); y = y_trval[idx]
    proxy = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000))
    p = cross_val_predict(proxy, EMB[idx], y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    t_bleached, t_healthy = p[y == 1].mean(), (1 - p[y == 0]).mean()       # per-class average self-confidence
    p_other = np.where(y == 1, 1 - p, p)
    flag = np.where(y == 1, (1 - p) >= t_healthy, p >= t_bleached)
    order = np.argsort(-p_other)
    drop = [i for i in order if flag[i]][: int(MAX_PRUNE * len(idx))]
    keep = np.setdiff1d(np.arange(len(idx)), drop)
    return idx[keep], idx[drop], roc_auc_score(y, p)

# scratch CNN baseline (same as Runs 2 and 3), trained with fixed epochs
def load_tensor(files, size=IMGSZ):
    return torch.stack([torch.from_numpy(np.array(Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR))).permute(2, 0, 1) for p in files])
X_trval, X_test = load_tensor(trval["img"].tolist()), load_tensor(test_files)

class SmallCNN(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                 nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, n_classes))
    def forward(self, x):
        return self.head(self.features(x))

def augment(xb):
    if random.random() < 0.5: xb = xb.flip(3)
    if random.random() < 0.5: xb = xb.flip(2)
    n = xb.shape[-1]; c = int(n * random.uniform(0.8, 1.0)); i, j = random.randint(0, n - c), random.randint(0, n - c)
    return F.interpolate(xb[..., i:i + c, j:j + c], size=(n, n), mode="bilinear", align_corners=False)

def cnn_fit(idx, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    X, y = X_trval[idx], torch.as_tensor(y_trval[idx])
    mean, std = (X.float() / 255).mean((0, 2, 3)), (X.float() / 255).std((0, 2, 3))      # training photos only
    net = SmallCNN().to(TDEV); opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CNN_EPOCHS)
    m_, s_ = mean.to(TDEV)[:, None, None], std.to(TDEV)[:, None, None]
    for ep in range(CNN_EPOCHS):
        net.train(); perm = torch.randperm(len(X))
        for k in range(0, len(perm), BATCH):
            b = perm[k:k + BATCH]
            if len(b) < 2: continue
            loss = F.cross_entropy(net(augment((X[b].to(TDEV).float() / 255 - m_) / s_)), y[b].to(TDEV))
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    net.eval(); net.norm = (mean, std)
    return net

def cnn_predict(net, X, bs=64):
    m_, s_ = net.norm[0].to(TDEV)[:, None, None], net.norm[1].to(TDEV)[:, None, None]
    with torch.no_grad():
        return torch.cat([net((X[k:k + bs].to(TDEV).float() / 255 - m_) / s_).softmax(1)[:, 1].cpu() for k in range(0, len(X), bs)]).numpy()
print("Feature matrix for the label cleaning:", EMB.shape)

## 3. Cross-validation: scratch CNN, A (reference), B (label cleaning)

All three use the Run 6 folds, so every comparison is paired. Progress prints after each fold.

In [ ]:
FOLD_ROOTS = [make_folder(f"fold{k}", tr, va) for k, (tr, va) in enumerate(FOLD_IDX)]
oof = {n: np.full(len(trval), np.nan) for n in ("CNN_baseline", "A_reference", "B_label_cleaning")}
minutes, prune_log, CLEAN_ROOTS = {n: 0.0 for n in oof}, [], {}
for k, (tr, va) in enumerate(FOLD_IDX):
    t0 = time.time(); oof["CNN_baseline"][va] = cnn_predict(cnn_fit(tr, SEED), X_trval[va]); minutes["CNN_baseline"] += (time.time() - t0) / 60

    t0 = time.time()
    oof["A_reference"][va] = yolo_predict(yolo_fit(FOLD_ROOTS[k], f"A_f{k}_s{SEED}", SEED), trval["img"].iloc[va].tolist())
    minutes["A_reference"] += (time.time() - t0) / 60

    t0 = time.time()
    keep, drop, proxy_auc = clean_training(tr)
    CLEAN_ROOTS[k] = make_folder(f"fold{k}_clean", keep, va)
    oof["B_label_cleaning"][va] = yolo_predict(yolo_fit(CLEAN_ROOTS[k], f"B_f{k}_s{SEED}", SEED), trval["img"].iloc[va].tolist())
    minutes["B_label_cleaning"] += (time.time() - t0) / 60
    prune_log.append(dict(fold=k, train=len(tr), dropped=len(drop), dropped_bleached=int(y_trval[drop].sum()),
                          dropped_healthy=int((y_trval[drop] == 0).sum()), proxy_auc=proxy_auc))
    print(f">>> fold {k + 1}/{FOLDS}: ROC AUC CNN {roc_auc_score(y_trval[va], oof['CNN_baseline'][va]):.3f}, "
          f"A {roc_auc_score(y_trval[va], oof['A_reference'][va]):.3f}, B {roc_auc_score(y_trval[va], oof['B_label_cleaning'][va]):.3f} "
          f"(B dropped {len(drop)} of {len(tr)} training photos)")
prune_log = pd.DataFrame(prune_log); prune_log.to_csv(os.path.join(OUT, "label_cleaning_by_fold.csv"), index=False)
print(prune_log.round(3).to_string(index=False))

## 4. Winner of A and B, then the 3-seed average in cross-validation

The better of A and B (out-of-fold macro F1 at its recall-floor cutoff, then ROC AUC) is trained again on every fold with seeds 43 and 44. Averaging the three seeds' probabilities gives the ensemble's out-of-fold predictions.

In [ ]:
def summarize(name, p):
    thr, reached = pick_threshold(y_trval, p); m5, mt = metrics_at(y_trval, p, 0.5), metrics_at(y_trval, p, thr)
    return dict(config=name, oof_roc_auc=m5["roc_auc"], macro_f1_at_0_5=m5["macro_f1"], recall_at_0_5=m5["bleached_recall"],
                cutoff=thr, recall_floor_reached=reached, macro_f1_at_cutoff=mt["macro_f1"], recall_at_cutoff=mt["bleached_recall"],
                healthy_recall_at_cutoff=mt["healthy_recall"], minutes=minutes.get(name, np.nan))

ab = pd.DataFrame([summarize(n, oof[n]) for n in ("A_reference", "B_label_cleaning")])
ab = ab.sort_values(["recall_floor_reached", "macro_f1_at_cutoff", "oof_roc_auc"], ascending=False)
WIN = ab.iloc[0]["config"]; print("Better single configuration:", WIN)

t0 = time.time(); seed_oof = {SEED: oof[WIN]}
for s in SEEDS[1:]:
    p = np.full(len(trval), np.nan)
    for k, (tr, va) in enumerate(FOLD_IDX):
        root = CLEAN_ROOTS[k] if WIN == "B_label_cleaning" else FOLD_ROOTS[k]
        p[va] = yolo_predict(yolo_fit(root, f"{WIN[0]}_f{k}_s{s}", s), trval["img"].iloc[va].tolist())
        print(f">>> {WIN} seed {s}, fold {k + 1}/{FOLDS} done ({(time.time() - t0) / 60:.1f} min)")
    seed_oof[s] = p
ENS = f"{WIN}_3seed_average"
oof[ENS] = np.mean([seed_oof[s] for s in SEEDS], axis=0); minutes[ENS] = minutes[WIN] + (time.time() - t0) / 60

cv = pd.DataFrame([summarize(n, p) for n, p in oof.items()])
cv["single_seed_auc_mean"] = [np.mean([roc_auc_score(y_trval, seed_oof[s]) for s in SEEDS]) if n == ENS else np.nan for n in cv["config"]]
cv.to_csv(os.path.join(OUT, "cv_results.csv"), index=False)
pd.DataFrame({"file": trval["file"], "true_label": y_trval, **oof, **{f"{WIN}_s{s}": v for s, v in seed_oof.items()}}).to_csv(os.path.join(OUT, "oof_predictions.csv"), index=False)
print(cv.drop(columns=["single_seed_auc_mean"]).round(3).to_string(index=False))

USE_ENS = cv.set_index("config").loc[ENS, "macro_f1_at_cutoff"] > cv.set_index("config").loc[WIN, "macro_f1_at_cutoff"]
FINAL_NAME = ENS if USE_ENS else WIN
THR = float(cv.set_index("config").loc[FINAL_NAME, "cutoff"])
THR_SINGLE = float(cv.set_index("config").loc[WIN, "cutoff"])
print(f"\nFinal model: {FINAL_NAME}; cutoff {THR:.3f} (single-seed cutoff {THR_SINGLE:.3f})")

fig, ax = plt.subplots(figsize=(5.2, 4.6)); markers = ["o", "s", "^", "D"]
for i, (n, p) in enumerate(oof.items()):
    fpr, tpr, _ = roc_curve(y_trval, p)
    ax.plot(fpr, tpr, color=COLORS[i], lw=2, marker=markers[i], markevery=0.1, ms=5, label=f"{n} (AUC {roc_auc_score(y_trval, p):.3f})")
ax.plot([0, 1], [0, 1], color="#888888", lw=1, ls="--"); ax.axhline(RECALL_MIN, color="#888888", lw=1, ls=":")
ax.set_xlabel("False positive rate (healthy flagged as bleached)"); ax.set_ylabel("Bleached recall")
ax.set_title(f"Run 7 out-of-fold ROC, {FOLDS}-fold CV (n={len(trval)})", fontsize=10)
ax.grid(alpha=0.25); ax.spines[["top", "right"]].set_visible(False); ax.legend(fontsize=7, frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "cv_roc.png"), dpi=150); plt.show()

## 5. Final models: trained on all 761 images (3 seeds), plus the scratch CNN

With label cleaning, the proxy cleans all 761 training photos the same way as in each fold. The test set has not been used yet.

In [ ]:
if WIN == "B_label_cleaning":
    keep_all, drop_all, _ = clean_training(np.arange(len(trval)))
    trval.iloc[drop_all][["file", "label"]].to_csv(os.path.join(OUT, "dropped_by_label_cleaning.csv"), index=False)
    print(f"Label cleaning dropped {len(drop_all)} of {len(trval)} photos ({int(y_trval[drop_all].sum())} bleached, {int((y_trval[drop_all] == 0).sum())} healthy)")
else:
    keep_all = np.arange(len(trval))
FULL_ROOT = make_folder("full", keep_all, FOLD_IDX[0][1])     # the val folder only satisfies Ultralytics; it is never scored
WEIGHTS = {}
for s in SEEDS:
    t0 = time.time(); w = yolo_fit(FULL_ROOT, f"final_s{s}", s)
    WEIGHTS[s] = os.path.join(OUT, f"final_model_s{s}.pt"); shutil.copy(w, WEIGHTS[s])
    print(f">>> final seed {s}: {(time.time() - t0) / 60:.1f} min")
cnn_final = cnn_fit(np.arange(len(trval)), SEED)

## 6. Held-out test set (evaluated once)

Each seed is scored at the single-model cutoff (mean ± std over 3 seeds, as in the proposal); the 3-seed average is scored at its own cutoff; everything is also scored at 0.5. The scratch CNN is scored at 0.5 for Objective 1.

In [ ]:
p_seed = {s: yolo_predict(WEIGHTS[s], test_files) for s in SEEDS}
p_ens = np.mean([p_seed[s] for s in SEEDS], axis=0)
p_cnn = cnn_predict(cnn_final, X_test)
rows = []
for s in SEEDS:
    rows += [dict(model=f"seed {s}", at="cutoff", threshold=THR_SINGLE, **metrics_at(y_test, p_seed[s], THR_SINGLE)),
             dict(model=f"seed {s}", at="0.5", threshold=0.5, **metrics_at(y_test, p_seed[s], 0.5))]
rows += [dict(model="3-seed average", at="cutoff", threshold=THR, **metrics_at(y_test, p_ens, THR)),
         dict(model="3-seed average", at="0.5", threshold=0.5, **metrics_at(y_test, p_ens, 0.5)),
         dict(model="scratch CNN", at="0.5", threshold=0.5, **metrics_at(y_test, p_cnn, 0.5))]
test_res = pd.DataFrame(rows); test_res.to_csv(os.path.join(OUT, "test_results.csv"), index=False)
pd.DataFrame(dict(file=test["file"], true_label=y_test, **{f"p_seed{s}": p_seed[s] for s in SEEDS}, p_average=p_ens, p_cnn=p_cnn)) \
  .to_csv(os.path.join(OUT, "test_predictions.csv"), index=False)
print(test_res.drop(columns=["cm"]).round(3).to_string(index=False))

def ms(frame, col): return frame[col].mean(), frame[col].std(ddof=1)
seeds_cut, seeds_05 = test_res[(test_res["model"].str.startswith("seed")) & (test_res["at"] == "cutoff")], test_res[(test_res["model"].str.startswith("seed")) & (test_res["at"] == "0.5")]
ENS_CUT = test_res[(test_res["model"] == "3-seed average") & (test_res["at"] == "cutoff")].iloc[0]
ENS_05 = test_res[(test_res["model"] == "3-seed average") & (test_res["at"] == "0.5")].iloc[0]
CNN_05 = test_res[test_res["model"] == "scratch CNN"].iloc[0]

## 7. External test set: BU + BHD (evaluated once)

Same rules as `eval_external.py`: labels and sources from the file names (BHD files are named `hbd`), dead corals and EILAT left out, exact repeats inside the set and matches with any Kaggle image (pHash distance of 5 or less) removed, photos padded to square.

In [ ]:
EXT = None
try:
    if EXT_SRC is None:
        from huggingface_hub import hf_hub_download
        EXT_SRC = hf_hub_download("esahit/coral-health-classification", "total_dataset.zip", repo_type="dataset")
    raw = os.path.join(BASE, "ext_raw"); shutil.rmtree(raw, ignore_errors=True)
    with zipfile.ZipFile(EXT_SRC) as z: z.extractall(raw)
    def parse(name):
        s = name.lower()
        lab = "bleached" if "unhealthy" in s else "healthy" if "healthy" in s else "dead" if "dead" in s else None
        src = "BHD" if ("hbd" in s or "bhd" in s) else "BU" if re.search(r"(?<![a-z])bu(?![a-z])", s) else "EILAT" if "eilat" in s else None
        return lab, src
    files = sorted(p for p in glob.glob(os.path.join(raw, "**", "*"), recursive=True) if p.lower().endswith(exts + (".tif", ".tiff")))
    ext = pd.DataFrame([(p, *parse(os.path.basename(p))) for p in files], columns=["path", "label", "source"])
    print(pd.crosstab(ext["source"].fillna("unknown"), ext["label"].fillna("unknown")))
    ext = ext[ext["label"].isin(["healthy", "bleached"]) & ext["source"].isin(["BU", "BHD"])].reset_index(drop=True)
    ext["phash"] = ext["path"].map(phash_of); ext = ext.dropna(subset=["phash"])
    n_raw = len(ext); ext = ext.drop_duplicates("phash").reset_index(drop=True); n_within = n_raw - len(ext)
    overlap = hamming(ext["phash"], df["phash"]).min(1) <= NEAR_DUP_T
    ext = ext[~overlap].reset_index(drop=True)
    sq = os.path.join(BASE, "ext_sq"); shutil.rmtree(sq, ignore_errors=True); os.makedirs(sq)
    ext["img"] = [os.path.join(sq, f"{r.source}_{i:05d}.jpg") for i, r in enumerate(ext.itertuples())]
    for r in ext.itertuples(): pad_square(r.path).save(r.img, quality=95)
    y_ext = (ext["label"] == "bleached").astype(int).to_numpy(); src = ext["source"].to_numpy()
    pe = {s: yolo_predict(WEIGHTS[s], ext["img"].tolist()) for s in SEEDS}; pe_ens = np.mean([pe[s] for s in SEEDS], axis=0)
    rows = []
    for name, mask in (("BU + BHD", np.ones(len(y_ext), bool)), ("BU", src == "BU"), ("BHD", src == "BHD")):
        for s in SEEDS:
            rows.append(dict(set=name, n=int(mask.sum()), model=f"seed {s}", threshold=THR_SINGLE, **metrics_at(y_ext[mask], pe[s][mask], THR_SINGLE)))
        rows.append(dict(set=name, n=int(mask.sum()), model="3-seed average", threshold=THR, **metrics_at(y_ext[mask], pe_ens[mask], THR)))
        rows.append(dict(set=name, n=int(mask.sum()), model="3-seed average @0.5", threshold=0.5, **metrics_at(y_ext[mask], pe_ens[mask], 0.5)))
    EXT = pd.DataFrame(rows); EXT.to_csv(os.path.join(OUT, "external_results.csv"), index=False)
    EXT_INFO = dict(n=len(y_ext), bleached=int(y_ext.sum()), healthy=int((y_ext == 0).sum()), removed_within=n_within, removed_overlap=int(overlap.sum()),
                    by_source=ext["source"].value_counts().to_dict())
    print(EXT_INFO); print(EXT.drop(columns=["cm"]).round(3).to_string(index=False))
except Exception as e:
    print("External test skipped:", repr(e))

## 8. Size and CPU latency

Mean over 100 test images, batch 1, after 10 warm-ups, image loading included. For the 3-seed average, the three models run one after another, so size and latency add up. Repeat on a team laptop for the final report.

In [ ]:
cpu_models = {s: YOLO(WEIGHTS[s]) for s in SEEDS}
def one(p, seeds):
    return [cpu_models[s].predict(p, imgsz=IMGSZ, device="cpu", verbose=False) for s in seeds]
LAT = {}
for label, seeds in (("single", [SEEDS[0]]), ("3-seed average", SEEDS)):
    for p in test_files[:10]: one(p, seeds)
    t0 = time.perf_counter()
    for p in test_files[:100]: one(p, seeds)
    LAT[label] = (time.perf_counter() - t0) / len(test_files[:100]) * 1000
SIZE = {"single": os.path.getsize(WEIGHTS[SEEDS[0]]) / 1e6, "3-seed average": sum(os.path.getsize(WEIGHTS[s]) for s in SEEDS) / 1e6}
card = dict(run=7, final=FINAL_NAME, models=[os.path.basename(WEIGHTS[s]) for s in (SEEDS if USE_ENS else [SEEDS[0]])],
            combine="mean of P(bleached)" if USE_ENS else "single model", threshold=THR if USE_ENS else THR_SINGLE,
            threshold_source=f"{FOLDS}-fold out-of-fold predictions on train + val (recall >= {RECALL_MIN})", imgsz=IMGSZ,
            classes=cpu_models[SEEDS[0]].names, label_cleaning=WIN == "B_label_cleaning", training=YOLO_AUG, epochs=EPOCHS,
            size_mb={k: round(v, 1) for k, v in SIZE.items()}, cpu_latency_ms={k: round(v) for k, v in LAT.items()})
json.dump(card, open(os.path.join(OUT, "final_model.json"), "w"), indent=2, default=str)
print("Size MB:", {k: round(v, 1) for k, v in SIZE.items()}, "| CPU latency ms:", {k: round(v) for k, v in LAT.items()})

## 9. Summary for the progress log

In [ ]:
f3 = lambda m: f"{m[0]:.3f} ± {m[1]:.3f}"
ok = lambda b: "met" if b else "NOT met"
final_cut = ENS_CUT if USE_ENS else None
L = [f"RUN 7 SUMMARY ({time.strftime('%b. %d, %Y')})",
     f"Data: cleaned split; train + validation {len(trval)} ({int(y_trval.sum())} bleached) in the Run 6 folds; test {len(test)} ({int(y_test.sum())} bleached).",
     "Label cleaning (training photos only), by fold: " + "; ".join(f"fold {r.fold}: dropped {r.dropped} of {r.train} ({r.dropped_bleached} bleached, {r.dropped_healthy} healthy), proxy AUC {r.proxy_auc:.3f}" for r in prune_log.itertuples()),
     f"Cross-validation (out-of-fold, n={len(trval)}):"]
for r in cv.itertuples():
    L.append(f"  {r.config}: ROC AUC {r.oof_roc_auc:.3f}; macro F1 @0.5 {r.macro_f1_at_0_5:.3f} (recall {r.recall_at_0_5:.3f}); cutoff {r.cutoff:.3f} -> "
             f"macro F1 {r.macro_f1_at_cutoff:.3f}, bleached recall {r.recall_at_cutoff:.3f}, healthy recall {r.healthy_recall_at_cutoff:.3f}")
L += [f"Better single configuration: {WIN}. Final model: {FINAL_NAME}; cutoff {THR:.3f} (single-seed cutoff {THR_SINGLE:.3f}).",
      f"Held-out test (n={len(test)}, evaluated once):",
      f"  Single models, seeds {SEEDS}, at cutoff {THR_SINGLE:.3f}: macro F1 {f3(ms(seeds_cut, 'macro_f1'))}, bleached recall {f3(ms(seeds_cut, 'bleached_recall'))}, "
      f"healthy recall {f3(ms(seeds_cut, 'healthy_recall'))}, ROC AUC {f3(ms(seeds_cut, 'roc_auc'))}.",
      f"  Single models at 0.5: macro F1 {f3(ms(seeds_05, 'macro_f1'))}, bleached recall {f3(ms(seeds_05, 'bleached_recall'))}.",
      f"  3-seed average at cutoff {THR:.3f}: macro F1 {ENS_CUT.macro_f1:.3f}, bleached recall {ENS_CUT.bleached_recall:.3f}, healthy recall {ENS_CUT.healthy_recall:.3f}, "
      f"ROC AUC {ENS_CUT.roc_auc:.3f}, confusion matrix {ENS_CUT.cm}.",
      f"  3-seed average at 0.5: macro F1 {ENS_05.macro_f1:.3f}, bleached recall {ENS_05.bleached_recall:.3f}, confusion matrix {ENS_05.cm}.",
      f"  Scratch CNN at 0.5: macro F1 {CNN_05.macro_f1:.3f}, bleached recall {CNN_05.bleached_recall:.3f}; "
      f"gain at 0.5: single models {100 * (ms(seeds_05, 'macro_f1')[0] - CNN_05.macro_f1):+.1f} points, 3-seed average {100 * (ENS_05.macro_f1 - CNN_05.macro_f1):+.1f} points."]
if EXT is not None:
    L.append(f"External BU + BHD (n={EXT_INFO['n']}: {EXT_INFO['bleached']} bleached, {EXT_INFO['healthy']} healthy; by source {EXT_INFO['by_source']}; "
             f"removed {EXT_INFO['removed_overlap']} Kaggle matches and {EXT_INFO['removed_within']} repeats; evaluated once):")
    for setname in ("BU + BHD", "BU", "BHD"):
        g = EXT[EXT["set"] == setname]; sd = g[g["model"].str.startswith("seed")]
        e = g[g["model"] == "3-seed average"].iloc[0]; e5 = g[g["model"] == "3-seed average @0.5"].iloc[0]
        L.append(f"  {setname} (n={int(g['n'].iloc[0])}): single models at cutoff macro F1 {f3(ms(sd, 'macro_f1'))}, recall {f3(ms(sd, 'bleached_recall'))}; "
                 f"3-seed average at cutoff macro F1 {e.macro_f1:.3f}, recall {e.bleached_recall:.3f}; at 0.5 macro F1 {e5.macro_f1:.3f}, recall {e5.bleached_recall:.3f}.")
else:
    L.append("External BU + BHD: skipped (see section 7).")
main = ENS_CUT if USE_ENS else None
mf1 = ENS_CUT.macro_f1 if USE_ENS else ms(seeds_cut, "macro_f1")[0]
mrec = ENS_CUT.bleached_recall if USE_ENS else ms(seeds_cut, "bleached_recall")[0]
key = "3-seed average" if USE_ENS else "single"
L += [f"Size {SIZE[key]:.1f} MB; CPU latency {LAT[key]:.0f} ms/image for the final model ({key}); single model {SIZE['single']:.1f} MB, {LAT['single']:.0f} ms (Colab CPU).",
      f"Targets on test (final model at its cutoff): macro F1 >= 0.90 {ok(mf1 >= 0.90)} ({mf1:.3f}); bleached recall >= 0.90 {ok(mrec >= 0.90)} ({mrec:.3f}); "
      f"size < 50 MB {ok(SIZE[key] < MAX_MB)}; latency < 200 ms {ok(LAT[key] < MAX_MS)}.",
      f"Environment: GPU {GPU_NAME}, ultralytics {__import__('ultralytics').__version__}, torch {torch.__version__}. Total time {(time.time() - T_START) / 60:.1f} min."]
summary = "\n".join(L); print(summary)
open(os.path.join(OUT, "run7_summary.txt"), "w").write(summary)
ZIP = shutil.make_archive(os.path.join(BASE, "run7_results"), "zip", OUT)
print("\nSaved", ZIP)
try:
    from google.colab import files
    files.download(ZIP)
except Exception:
    pass